___
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://sodal.cl/wp-content/uploads/2024/08/caracteristicas-aluminio-600x375.jpg" width="380px" height="200px" />


# <font color= #bbc28d> **Sistema Híbrido de Predicción y Gestión de Riesgo (FNN + GARCH)** </font>
#### <font color= #2E9AFE> `Examen 2 - MNLP`</font>
- <Strong> Diana Valdivia, Daniela de la Torre, Aissa Gonzáles & Rafael Takata. </Strong>
- <Strong> Fecha </Strong>: 12/04/2026.

___

<p style="text-align:right;"> Image retrieved from: https://sodal.cl/wp-content/uploads/2024/08/caracteristicas-aluminio-600x375.jpg</p>

El activo elegido para este estudio es el aluminio, un commodity industrial de gran relevancia por su amplia utilización en sectores como la construcción, el transporte, el empaque y la manufactura. Su precio está influenciado por factores como la oferta global, la demanda industrial, los costos energéticos y las condiciones macroeconómicas, lo que lo convierte en un activo interesante para analizar desde una perspectiva financiera y de riesgo. Además, al tratarse de un commodity con variaciones frecuentes en su precio, representa un buen caso de estudio para aplicar modelos predictivos y de volatilidad. La motivación principal de este trabajo es mostrar cómo un Hedge Fund podría usar herramientas FFNN y GARCH para apoyar sus decisiones de inversión.

Dataset = `https://www.investing.com/commodities/aluminum-historical-data`

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import acf, pacf
from scipy import stats
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

# Renderizado HTML en Jupyter / VS Code
pio.renderers.default = "notebook"
#pio.renderers.default = "colab"

# **<font color= #bb90ce> Lectura y Procesamiento de datos </font>**

In [ ]:
datos = pd.read_csv(r"Aluminium Historical Data_today.csv")
datos = datos[['Date', 'Price']].copy()

datos['Date'] = pd.to_datetime(datos['Date'], format='mixed')
datos['Price'] = datos['Price'].str.replace(',', '').astype(float)

datos = datos.sort_values('Date')

fig = px.line(datos, x='Date', y='Price', title='Precio del Aluminio')
fig.show()

A simple vista, se puede ver como el precio del aluminio ha ido subiendo desde el último año y se ha mantendio con ujna tendencia a la alta muy marcada. Podemos ver como su pico se dio a inicios del 2022, siendo el histórico máximo de los últimos 4 años.

In [ ]:
# Parsear fecha y ordenar cronológicamente
datos = datos.sort_values('Date').reset_index(drop=True)
datos.set_index('Date', inplace=True)

print("=" * 55)
print("  RESUMEN GENERAL DEL DATASET")
print("=" * 55)
print(f"  Periodo     : {datos.index.min().date()} → {datos.index.max().date()}")
print(f"  Observaciones: {len(datos):,}")
print(f"  Frecuencia aprox.: {pd.infer_freq(datos.index) or 'irregular (ver gaps)'}")

# Valores nulos
nulos = datos.isnull().sum()
print(f"  Valores nulos: {nulos.values[0]}")

  RESUMEN GENERAL DEL DATASET
  Periodo     : 2022-01-04 → 2026-04-10
  Observaciones: 1,078
  Frecuencia aprox.: irregular (ver gaps)
  Valores nulos: 0


En nuestro caso, como en la mayoría de las comodities, la frecuencia de la serie es de tipo `B`, es decir, los días hábiles de lunes a viernes ecceptuando algunos días festivos. Para poder trabajar con la serie se respetara este orden temporal y se rellenaran los huecos dentro de los días festivos:

In [ ]:
fechas_completas = pd.date_range(datos.index.min(), datos.index.max())
faltantes = fechas_completas.difference(datos.index)

print(len(faltantes))
print(faltantes[:10])

480
DatetimeIndex(['2022-01-08', '2022-01-09', '2022-01-15', '2022-01-16',
               '2022-01-22', '2022-01-23', '2022-01-29', '2022-01-30',
               '2022-02-05', '2022-02-06'],
              dtype='datetime64[us]', freq=None)


En total, tenemos 480 registros que hacen falta, la mayoría de estos fines de semana:

In [ ]:
# Rellenamos business days faltantes
datos = datos.asfreq('B').ffill()

fechas_completas = pd.date_range(datos.index.min(),datos.index.max(),freq='B')
faltantes = fechas_completas.difference(datos.index)

print(len(faltantes))

0


Una vez listo el procesamiento general, pasaremos a los dos distintos tipos de modelado que haremos en este Examen: **GARCH** y **FFNN**. Cada una con sus distintos pasos y metodologías:

# **<font color= #bbc28d> GARCH </font>**
El modelo **GARCH (Generalized Autoregressive Conditional Heteroskedasticity)** es una técnica utilizada para modelar y predecir la **volatilidad** en series de tiempo financieras, como precios de commodities, acciones o divisas.

A diferencia de modelos tradicionales que asumen varianza constante, GARCH permite que la **varianza cambie en el tiempo**, lo cual es más realista para datos financieros donde existen periodos de alta y baja volatilidad.

### **Idea principal**
- Los retornos financieros suelen mostrar **clustering de volatilidad**
- Periodos con alta volatilidad tienden a ser seguidos por alta volatilidad
- Periodos tranquilos tienden a permanecer tranquilos

GARCH modela este comportamiento haciendo que la varianza dependa de:
1. Errores pasados (shocks recientes)
2. Varianza pasada (persistencia)

### **¿Por qué usarlo en el precio del aluminio?**
El precio del aluminio, como otros commodities, presenta:
- Periodos de alta volatilidad
- Eventos macroeconómicos
- Choques de oferta y demanda

GARCH permite capturar estos cambios dinámicos en la volatilidad y generar predicciones más realistas.



## **<font color= #bbc28d> &ensp; • Obtenemos datos y gráficas de serie de tiempo </font>**
Obtenemos la serie de tiempo del sitio web ìnvesting.com` para graficar. Se calcula los rendimientos logarítmicos porcentuales, los cuales se representan como:$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) \times 100$$

Usualmente se usan retorno logaritmico porque en escalas pequeñas su compertamiento es muy similar al del retorno "normal" y al ser logaritmo, el comportamiento sigue una distribución más estable.

In [ ]:
# Calcular retornos con los datos filtrados
returns = 100 * np.log(datos['Price'] / datos['Price'].shift(1)).dropna()

# Gráfica de la serie de retornos usando Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines', name='Retornos MAL3'))
fig.update_layout(title=f'Retornos Diarios del Aluminio',
                  yaxis_title='Retornos (%)')
fig.show()

## **<font color= #bbc28d> &ensp; • Selección y validación de modelo. </font>**

Para justificar si es estadísticamente viable utilizar un modelo GARCH, se calculan las graficas ACF y PACF; pero utilizando los retornos al cuadrado.

**Retornos al Cuadrado**

Las gráficas ACF y PACF miden la autocorrelación (qué tanto se parece el dato de hoy vs los datos del pasado)
- ACF/PACF de los Retornos al Cuadrado ($r_t^2$)
  - ¿Qué miden? ->
    Dependencia no lineal en el segundo momento estadístico (la varianza/volatilidad). Al elevar al cuadrado, eliminamos el signo (positivo o negativo) y nos quedamos solo con la magnitud del movimiento.
  - ¿Qué pregunta responden? ->
    Saber que ayer hubo un movimiento violento (sin importar si fue hacia arriba o hacia abajo) me ayuda a predecir si hoy el mercado seguirá turbulento.
  - Lo que verás en la gráfica ->
    Verás múltiples barras que sobresalen de la zona de significancia. Esto demuestra matemáticamente el Volatility Clustering (agrupamiento de volatilidad).

Si la PACF o ACF de retornos al cuadrado no muestra barras significativas, no se recomienda utilizar un modelo GARCH (Puede ser un modelo EGARCH o de otra familia)

In [ ]:
# Retornos al cuadrado
sq_returns = returns**2

# Calcular ACF y PACF
lag_acf = acf(sq_returns, nlags=20)
lag_pacf = pacf(sq_returns, nlags=20, method='ols')

# Graficar ACF y PACF con Plotly
fig = make_subplots(rows=1, cols=2, subplot_titles=('ACF de Retornos al Cuadrado', 'PACF de Retornos al Cuadrado'))

# Añadir barras de ACF
fig.add_trace(go.Bar(x=np.arange(len(lag_acf)), y=lag_acf, name='ACF'), row=1, col=1)
# Añadir barras de PACF
fig.add_trace(go.Bar(x=np.arange(len(lag_pacf)), y=lag_pacf, name='PACF'), row=1, col=2)

# Líneas de significancia (aprox 1.96 / sqrt(N))
sig_level = 1.96 / np.sqrt(len(returns))
for i in [1, 2]:
    fig.add_hline(y=sig_level, line_dash="dash", line_color="red", row=1, col=i)
    fig.add_hline(y=-sig_level, line_dash="dash", line_color="red", row=1, col=i)

fig.update_layout(title='Análisis de Dependencia de Varianza', showlegend=False)
fig.show()

Siguiendo el principio de parsimonia, elegimos el modelo **GARCH(1,1)** porque es el más simple dentro de los modelos GARCH, pero al mismo tiempo es uno de los más utilizados en finanzas.

Este modelo permite capturar la característica más importante de los retornos financieros: la volatilidad cambiante en el tiempo (periodos de alta y baja volatilidad).

El **GARCH(1,1)** usa únicamente dos componentes: la volatilidad pasada y los choques recientes, lo que lo hace fácil de interpretar y suficientemente potente para modelar la mayoría de series financieras, como los retornos del aluminio.

Por estas razones, es un buen punto de partida para estimar la volatilidad y construir medidas de riesgo como el VaR.

## **<font color= #bbc28d> &ensp; • Modelado </font>**

### **Forma general del modelo GARCH(1,1)**

La varianza condicional se define como:

$
\sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2
$

Donde:
- $ \sigma_t^2 $ : varianza condicional en el tiempo t
- $ \omega $ : constante
- $ \alpha $ : efecto de shocks recientes
- $ \beta $ : persistencia de la volatilidad
- $ \varepsilon_{t-1}^2 $ : error cuadrado del periodo anterior

In [ ]:
# Ajustar modelo GARCH(1,1) con distribución Normal
model = arch_model(returns, vol='Garch', p=1, q=1, dist='Normal')
garch_fit = model.fit(disp='off')

# Extraer volatilidad condicional
cond_vol = garch_fit.conditional_volatility

# Cuantil empírico al 5% de los residuos estandarizados
q_5 = garch_fit.std_resid.quantile(0.05)

# VaR = Media Condicional + (Cuantil * Volatilidad Condicional)
mean = garch_fit.params['mu']
VaR_5 = mean + cond_vol * q_5

# Contar violaciones (exceedances)
violaciones = (returns < VaR_5).sum()
tasa_violacion = violaciones / len(returns) * 100

# Resumen del modelo
print("="*84)
print("MODELO GARCH(1,1) CON DISTRIBUCIÓN NORMAL")
print("="*84)
print(garch_fit.summary())

MODELO GARCH(1,1) CON DISTRIBUCIÓN NORMAL
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  Price   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -1869.56
Distribution:                  Normal   AIC:                           3747.13
Method:            Maximum Likelihood   BIC:                           3767.19
                                        No. Observations:                 1113
Date:                Sun, Apr 12 2026   Df Residuals:                     1112
Time:                        18:45:48   Df Model:                            1
                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
mu             0.04

El modelo GARCH(1,1) muestra que la volatilidad de los precios del aluminio presenta **persistencia y dependencia** temporal, lo cual es característico de series financieras.

Los parámetros α (alpha) y β (beta) son estadísticamente **significativos**, y su suma cercana a uno indica que los periodos de alta volatilidad tienden a mantenerse durante varios días, reflejando un fuerte volatility clustering.

El parámetro β ≈ 0.94 sugiere una alta **persistencia** en la volatilidad, lo que implica que los choques en el mercado **tardan** en disiparse.

Por otro lado, el parámetro de media μ no es estadísticamente significativo, lo cual es consistente con la teoría financiera, donde los retornos suelen ser difíciles de predecir en su valor esperado.

In [ ]:
# Estadísticas del VaR
print("\n" + "="*50)
print("ESTADÍSTICAS DEL VAR 5%")
print("="*50)
print(f"Violaciones observadas: {violaciones} ({tasa_violacion:.2f}%)")
print(f"Violaciones esperadas: {len(returns) * 0.05:.0f} (5.00%)")
print(f"VaR promedio: {VaR_5.mean():.4f}%")
print(f"VaR mínimo: {VaR_5.min():.4f}%")
print(f"VaR máximo: {VaR_5.max():.4f}%")


ESTADÍSTICAS DEL VAR 5%
Violaciones observadas: 56 (5.03%)
Violaciones esperadas: 56 (5.00%)
VaR promedio: -2.0551%
VaR mínimo: -4.3189%
VaR máximo: -1.2268%


Podemos ver que de acuerdo con el citerio del 5%, nuestro modelo esta justo en el limite de las violaciones de un modelo robusto por lo que podríamos considerar que es bueno generalizando.

In [ ]:
# Graficar Retornos vs VaR
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=returns.index,
    y=returns.squeeze(),
    mode='lines',
    name='Retornos Reales',
    line=dict(color='blue'),
    opacity=0.7
))

fig.add_trace(go.Scatter(
    x=VaR_5.index,
    y=VaR_5,
    mode='lines',
    name='VaR 5% (GARCH)',
    line=dict(color='red', width=2)
))

# Marcar violaciones
violaciones_idx = returns[returns < VaR_5].index
violaciones_val = returns[returns < VaR_5].values

fig.add_trace(go.Scatter(
    x=violaciones_idx,
    y=violaciones_val,
    mode='markers',
    name=f'Violaciones ({violaciones})',
    marker=dict(color='green', size=6, symbol='circle')
))

fig.update_layout(
    title=f'Backtesting VaR 5% GARCH(1,1) - Aluminio<br>Violaciones: {violaciones}/{len(returns)} ({tasa_violacion:.2f}%)',
    xaxis_title='Fecha',
    yaxis_title='Retornos (%)',
    hovermode='x unified',
    height=500
)

fig.show()

## **<font color= #bbc28d> &ensp; • Half-life de un shock en un modelo GARCH</font>**
Debido a la naturaleza de los componentes autorregresivos en los modelos **GARCH**, los choques se disipan a una **tasa exponencial** y **nunca desaparecen completamente**.

Por lo tanto, que un choque se disipe totalmente **tomaría un tiempo infinito**, y la **varianza de largo plazo nunca se alcanza exactamente**, sino que únicamente se **aproxima cada vez más**.

Por esta razón, en lugar de estudiar cuándo desaparece completamente un choque, se analiza la **vida media (half-life)** del shock.

La fórmula general es:

$$
\ell := \frac{\ln(0.5)}{\ln\left(\sum_{i=1}^{s}\alpha_i + \sum_{j=1}^{r}\beta_j\right)}
$$

donde:

- $\alpha_i$ son los coeficientes **ARCH**
- $\beta_j$ son los coeficientes **GARCH**

En el caso particular de un **GARCH(1,1)**, la fórmula se simplifica a:

$$
\ell = \frac{\ln(0.5)}{\ln(\alpha_1 + \beta_1)}
$$

La **vida media** indica **después de cuántos periodos la mitad del efecto de un choque en la volatilidad se ha disipado**.

In [ ]:
# Parámetros del modelo Garch(1,1)
alpha = garch_fit.params['alpha[1]']
beta = garch_fit.params['beta[1]']

# Calcular la vida media de choque.
half_life = np.log(0.5) / np.log(alpha + beta)

print("Vida Media del Choque:", half_life)

Vida Media del Choque: 77.01129902385351


Para el modelo GARCH(1,1) estimado, la vida media es aproximadamente **77** días, lo que sugiere que los shocks en la volatilidad de la acción tardan alrededor de dos meses y medio de mercado en disiparse parcialmente.

Una vez realizado el modelo, es necesario realizar los forecast para la semana del **13 al 17 de abri**l:

In [ ]:
forecast = garch_fit.forecast(horizon=5)

# Volatilidad (raíz de la varianza)
vol_forecast = np.sqrt(forecast.variance.values[-1, :])

# Calcular VaR para los próximos días
VaR_forecast = mean + vol_forecast * q_5

# Fechas objetivo
dates = pd.date_range(start="2026-04-13", periods=5)

# Crear DF de predicciones
results = pd.DataFrame({
    'Date': dates,
    'Volatility (%)': vol_forecast,
    'VaR_95 (%)': VaR_forecast

})

results

,Date,Volatility (%),VaR_95 (%)
0,2026-04-13,1.702741,-2.612187
1,2026-04-14,1.700640,-2.608909
2,2026-04-15,1.698556,-2.605656
3,2026-04-16,1.696487,-2.602428
4,2026-04-17,1.694435,-2.599225


In [ ]:
# Guardamos los resultados
results.to_csv(r"Forecast_GARCH.csv", index=False)

# **<font color= #66b0b0> FFNN </font>**
Una **FFNN (Feedforward Neural Network)** es un tipo de red neuronal artificial utilizada para modelar relaciones no lineales entre variables. Se denomina *feedforward* porque la información fluye en una sola dirección: desde la capa de entrada, pasando por capas ocultas, hasta la capa de salida, sin ciclos ni retroalimentación.

Estas redes son especialmente útiles cuando la relación entre variables es compleja y difícil de modelar con técnicas estadísticas tradicionales.

### **¿Para qué se usa en series financieras?**

Las FFNN pueden:

* Capturar relaciones no lineales
* Detectar patrones complejos
* Aprender dinámicas temporales usando *lags*

### **Aplicación al precio del aluminio**

Para el precio del aluminio, una FFNN puede:

* Aprender patrones ocultos en la serie
* Modelar dinámicas no lineales del mercado



## **<font color= #66b0b0> &ensp; • Log Returns vs Precio de Cierre</font>**

En el modelado de series financieras o de commodities como el aluminio, la elección de la variable objetivo es crítica para el desempeño de modelos de aprendizaje automático como las redes neuronales feedforward (FFNN). En particular, cuando la serie presenta una tendencia fuertemente creciente en el tiempo, como ocurre en el precio del aluminio en el último año, el uso de log-returns resulta metodológicamente más apropiado que el uso del nivel de precios, ya que estos reducen la influencia del nivel de la serie y reexpresan la información en términos de variaciones relativas entre periodos, generando una señal más homogénea y evitando que el modelo se enfoque principalmente en la tendencia de largo plazo.

En este contexto, una FFNN entrenada directamente sobre precios tiende a dar un peso excesivo a la estructura de tendencia, lo que puede llevar a que el modelo minimice la función de pérdida principalmente ajustando el crecimiento del nivel, en lugar de capturar dinámicas relevantes de corto plazo como choques de oferta y demanda o cambios en la volatilidad, generando así un problema de sobreajuste a la estructura de nivel.

Dado que las FFNN no incorporan explícitamente estructura temporal, su desempeño depende en gran medida de la calidad estadística de la variable objetivo; por ello, el uso de log-returns contribuye a reducir problemas asociados a escalas variables, mejora la estabilidad numérica del entrenamiento y favorece una mejor generalización fuera de muestra, mientras que el uso de precios puede introducir heterocedasticidad inducida por el nivel y sensibilidad excesiva a periodos recientes de crecimiento acelerado.


In [ ]:
datos['LogReturn'] = np.log(datos['Price'] / datos['Price'].shift(1))
datos = datos.dropna()

serie = datos[['LogReturn']]

Una vez realizada nuestra variable a predecir, hacemos nuestro train test split para evitar data leakage, aunque dentro e este contexto, al ser a futuro, no estamos en sí haciendolo pero igual por práctica lo respetaremos:

In [ ]:
# Train/Val/Test split
WINDOW_SIZE = 15
TEST_SIZE   = 50

n          = len(serie)
train_size = int((n - TEST_SIZE) * 0.75)
val_size   = (n - TEST_SIZE) - train_size

train = serie.iloc[:train_size]
val   = serie.iloc[train_size:train_size + val_size]
test  = serie.iloc[-TEST_SIZE:]

# Escalar variables
scaler       = MinMaxScaler(feature_range=(-1, 1))
train_scaled = scaler.fit_transform(train)   # ← fit sobre retornos de train
val_scaled   = scaler.transform(val)
test_scaled  = scaler.transform(test)

# Crear ventanas
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size, 0])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

X_train, y_train = crear_ventanas(train_scaled, WINDOW_SIZE)
X_val,   y_val   = crear_ventanas(val_scaled,   WINDOW_SIZE)
test_ctx         = np.vstack([val_scaled[-WINDOW_SIZE:], test_scaled]) # Con info real del val split
X_test,  y_test  = crear_ventanas(test_ctx, WINDOW_SIZE)

## <font color= #66b0b0> &ensp; • **Modelado** </font>
Una red neuronal feedforward se compone de:

* **Capa de entrada**: recibe los datos (por ejemplo, retornos pasados)
* **Capas ocultas**: transforman la información mediante funciones no lineales
* **Capa de salida**: genera la predicción final

Cada neurona aplica la siguiente transformación:

$
y = f\left(\sum_{i=1}^{n} w_i x_i + b \right)
$

Donde:

* $x_i$ : variables de entrada
* $w_i$ : pesos de la red
* $b$ : sesgo (bias)
* $f$ : función de activación (ReLU, tanh, sigmoid, etc.)

## **<font color= #66b0b0> &ensp; • *Directional Loss*</font>**

La función de pérdida propuesta combina un término clásico de error cuadrático medio (MSE) con una penalización adicional asociada a la **dirección del movimiento** predicho.

En primer lugar, el componente de MSE permite controlar la precisión en términos de magnitud, asegurando que las predicciones se mantengan cercanas a los valores reales. Sin embargo, en problemas de series temporales financieras, no solo importa el tamaño del error, sino también la **capacidad del modelo para anticipar correctamente la dirección del cambio** (positivo o negativo), ya que esta información es crítica en decisiones de trading, cobertura o gestión de riesgo.

Por esta razón, se introduce un término adicional de penalización que incrementa la pérdida cuando el modelo predice una dirección incorrecta respecto al valor real. Esta penalización se activa cuando el signo de la predicción difiere del signo del valor observado, y además pondera este error en función de su magnitud, lo que evita penalizar en exceso fluctuaciones pequeñas consideradas ruido.

La combinación de ambos términos permite equilibrar dos objetivos: por un lado, mantener una buena aproximación numérica de la serie y, por otro, incentivar que el modelo aprenda la estructura direccional del mercado. El factor de ponderación ajusta la importancia relativa de la direccionalidad frente al error en magnitud, permitiendo controlar el grado de sensibilidad del modelo a cambios de signo.

En conjunto, esta función de pérdida resulta adecuada para escenarios donde la **dirección del movimiento es tan importante como su tamaño**, como ocurre frecuentemente en el modelado de retornos de activos financieros o commodities.

In [ ]:
# Establecer arquitectura del modelo
model = Sequential([
    Dense(64, activation='relu', input_dim=WINDOW_SIZE),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

# Funcion de pérdida para penalizar DA incorrecto
def directional_loss(y_true, y_pred):
    # MSE base — controla magnitud
    mse = tf.reduce_mean(tf.square(y_true - y_pred))

    # Penalización cuando el signo es incorrecto
    sign_true = tf.sign(y_true)
    sign_pred = tf.sign(y_pred)
    wrong_dir = tf.cast(tf.not_equal(sign_true, sign_pred), tf.float32)
    penalty   = tf.reduce_mean(wrong_dir * tf.square(y_true - y_pred))

    return mse + 0.5 * penalty

model.compile(optimizer=Adam(learning_rate=0.001),loss=directional_loss,metrics=['mae'])

callbacks = [
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
]

# Minimizar sobre el DA de los retornos
history = model.fit(X_train, y_train,  epochs=100, batch_size=16,validation_data=(X_val, y_val), callbacks=callbacks,verbose=1)


Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - loss: 0.2462 - mae: 0.3301 - val_loss: 0.1859 - val_mae: 0.3141 - learning_rate: 0.0010
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1977 - mae: 0.3157 - val_loss: 0.1645 - val_mae: 0.2977 - learning_rate: 0.0010
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1765 - mae: 0.2951 - val_loss: 0.1637 - val_mae: 0.2966 - learning_rate: 0.0010
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1520 - mae: 0.2822 - val_loss: 0.1657 - val_mae: 0.3015 - learning_rate: 0.0010
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1524 - mae: 0.2739 - val_loss: 0.1695 - val_mae: 0.3051 - learning_rate: 0.0010
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1340 - mae: 0.2687 - val_loss: 0.1679 - val_mae: 0.3026 - learning_rate: 0.0010
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1329 - mae: 0.2632 - val_loss: 0.1650 - val_mae: 0.2996 - learning_rate: 0.0010
Epoch 8/100
9/9 ━━━━

In [ ]:
# Predicción y recontrucción de precios
y_pred_scaled = model.predict(X_test, verbose=0)

# Invertir escalado → log-returns en escala real
y_pred_lr = scaler.inverse_transform(y_pred_scaled).ravel()
y_test_lr = scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()

# Reconstruir precios: P_t = P_{t-1} * exp(lr_pred)
idx_inicio      = train_size + val_size
precio_anterior = datos['Price'].iloc[idx_inicio - 1 : idx_inicio - 1 + len(y_pred_lr)].values
y_pred_precio   = precio_anterior * np.exp(y_pred_lr)
y_real_precio   = datos['Price'].iloc[idx_inicio : idx_inicio + len(y_pred_lr)].values

# Calcular métricas
mae  = mean_absolute_error(y_real_precio, y_pred_precio)
rmse = np.sqrt(mean_squared_error(y_real_precio, y_pred_precio))
mape = np.mean(np.abs((y_real_precio - y_pred_precio) / y_real_precio)) * 100
da   = np.mean(np.sign(y_test_lr) == np.sign(y_pred_lr)) * 100

print(f"MAE  : {mae:.2f} USD/t")
print(f"RMSE : {rmse:.2f} USD/t")
print(f"MAPE : {mape:.2f}%")
print(f"DA   : {da:.1f}%")

# Gráficar
test_index      = datos.index[idx_inicio : idx_inicio + len(y_pred_lr)]
historico_index = datos.index[:idx_inicio]

fig = go.Figure()
fig.add_trace(go.Scatter(x=historico_index,          y=datos['Price'].iloc[:idx_inicio], name='Histórico'))
fig.add_trace(go.Scatter(x=test_index,               y=y_real_precio,                    name='Test Real'))
fig.add_trace(go.Scatter(x=test_index,               y=y_pred_precio,                    name='Predicción'))
fig.update_layout(title='FFNN MAL3 — Real vs Predicción (precios reconstruidos)')
fig.show()

MAE  : 44.81 USD/t
RMSE : 62.04 USD/t
MAPE : 1.37%
DA   : 62.0%


El modelo presenta un desempeño razonable en términos generales, con un MAE de 4**4.81 USD** y un RMSE de **62.04 USD,** lo que indica que los errores promedio son moderados pero con presencia de algunos errores grandes ocasionales. El MAPE de **1.37%** sugiere una buena precisión relativa en la predicción del nivel de precios, aunque esta métrica puede verse optimista en series con precios altos o relativamente estables. Por otro lado, una directional accuracy de **62%** indica que el modelo logra capturar parcialmente la dirección de los movimientos del precio, superando el azar pero aún con margen importante de mejora. En conjunto, el modelo parece desempeñarse mejor en la predicción del nivel del precio que en la captura consistente de la dinámica direccional del mercado.

Para ver si nuestro modelo no esta sobre-ajustado, podemos hacer un pequeño ejercicio, nuestro dataset lo dejamos hasta el día 2 de abril por lo que la semana del 6 al 10 de abril son datos que aún no ha visto y que tal vez nos podrían ayudar a ver si hay algún problema de overfitting, por lo que podemos comprobar las métricas similar a como sería en la semana del 13 a 17.

In [ ]:
# Leemos dataset con datos de mitades de marzo a abril 10
datos = pd.read_csv("Aluminium Historical Data Test.csv")

# Mismo procesamiento de antes
datos = datos[['Date', 'Price']].copy()
datos['Date']  = pd.to_datetime(datos['Date'], format='mixed')
datos['Price'] = datos['Price'].str.replace(',', '').astype(float)
datos = datos.sort_values('Date').reset_index(drop=True)
datos.set_index('Date', inplace=True)
datos = datos.asfreq('B')
datos['Price'] = datos['Price'].ffill()


print(f"Aluminio — Periodo : {datos.index.min().date()} → {datos.index.max().date()}")
print(f"Aluminio — Filas   : {len(datos)}")

In [ ]:
# Calculamos los retornos
datos['LogReturn'] = np.log(datos['Price'] / datos['Price'].shift(1))

# Fecha de corte de la serie hasta el último viernes antes del 6
fecha_corte = "2026-04-03"
datos_cut = datos.loc[:fecha_corte].copy()
serie_cut = datos_cut[['LogReturn']]
all_scaled = scaler.transform(serie_cut)

ultimo_precio = datos_cut['Price'].iloc[-1]

In [ ]:
def forecast_5_dias(model, all_scaled, window_size, scaler, ultimo_precio):
    contexto = list(all_scaled[-window_size:, 0])
    preds_lr = []

    for _ in range(5):
        X_input     = np.array(contexto[-window_size:]).reshape(1, -1)
        pred_scaled = model.predict(X_input, verbose=0)[0][0]
        pred_lr     = scaler.inverse_transform([[pred_scaled]])[0][0]
        preds_lr.append(pred_lr)
        contexto.append(pred_scaled)

    precios = []
    p = ultimo_precio
    for lr in preds_lr:
        p = p * np.exp(lr)
        precios.append(p)

    return np.array(preds_lr), np.array(precios)

preds_lr_f, precios_f = forecast_5_dias(
    model,
    all_scaled,
    15,
    scaler,
    ultimo_precio
)

fechas_f = pd.bdate_range(
    start="2026-04-06",
    end="2026-04-10"
)

df_forecast = pd.DataFrame({
    'Precio_pred': precios_f.round(2),
    'LogReturn'  : preds_lr_f.round(5),
    'Direccion'  : ['↑' if r > 0 else '↓' for r in preds_lr_f]
}, index=fechas_f)

print("\nForecast próximos 5 días hábiles:")
print(df_forecast.to_string())
real = datos.loc[fechas_f]['Price']

real_dir = np.sign(np.diff(real.values))
pred_dir = np.sign(np.diff(precios_f))

DA = np.mean(real_dir == pred_dir)

print("Directional Accuracy:", round(DA,3))
print("Directional Accuracy %:", round(DA*100,2))
mae = np.mean(np.abs(real.values - precios_f))
rmse = np.sqrt(np.mean((real.values - precios_f)**2))

print("MAE:", mae)
print("RMSE:", rmse)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=real.index,
    y=real.values,
    name="Real",
    line=dict(color="black", width=3)
))

fig.add_trace(go.Scatter(
    x=df_forecast.index,
    y=df_forecast['Precio_pred'],
    name="Predicción",
    mode="lines+markers",
    line=dict(color="red", dash="dash")
))

fig.update_layout(
    title="FFNN Returns — Forecast vs Real (6-10 Abril 2026)",
    xaxis_title="Fecha",
    yaxis_title="Precio"
)

fig.show()


Forecast próximos 5 días hábiles:
            Precio_pred  LogReturn Direccion
2026-04-06      3475.05    0.00189         ↑
2026-04-07      3479.12    0.00117         ↑
2026-04-08      3488.99    0.00283         ↑
2026-04-09      3465.68   -0.00670         ↓
2026-04-10      3474.06    0.00241         ↑
Directional Accuracy: 0.75
Directional Accuracy %: 75.0
MAE: 17.687116298735965
RMSE: 20.61422304820894


Podemos ver que el DA mínimo de esa semana, supera el 55%, por lo que podemos realizar predicciones para la proxima semana:

In [ ]:
preds_lr_f, precios_f = forecast_5_dias(model, all_scaled, 15, scaler, ultimo_precio)

fechas_f = pd.bdate_range(start=datos.index[-1] + pd.offsets.BDay(1), periods=5)

df_forecast = pd.DataFrame({
    'Precio_pred': precios_f.round(2),
    'LogReturn'  : preds_lr_f.round(5),
    'Direccion'  : ['↑' if r > 0 else '↓' for r in preds_lr_f]
}, index=fechas_f)

print("\nForecast próximos 5 días hábiles:")
print(df_forecast.to_string())

# Gráfica forecast
ultimos_60 = datos['Price'].iloc[-60:]
fig_f = go.Figure()
fig_f.add_trace(go.Scatter(x=ultimos_60.index, y=ultimos_60.values, name='Histórico reciente'))
fig_f.add_trace(go.Scatter(x=df_forecast.index, y=df_forecast['Precio_pred'],
    name='Forecast 5 días', mode='lines+markers',
    line=dict(color='red', dash='dash'), marker=dict(size=9)))
fig_f.update_layout(title='FFNN MAL3 — Forecast próximos 5 días hábiles')
fig_f.show()


Forecast próximos 5 días hábiles:
            Precio_pred  LogReturn Direccion
2026-04-13      3502.41   -0.00252         ↓
2026-04-14      3493.23   -0.00262         ↓
2026-04-15      3494.85    0.00046         ↑
2026-04-16      3497.45    0.00074         ↑
2026-04-17      3520.18    0.00648         ↑


In [ ]:
# Guardar modelo entero
model.save("modelo_ffnn_ret.keras")

# Guardar el scaler utilizado
joblib.dump(scaler, "scaler_ffnn_ret.pkl")

['scaler_ffnn.pkl']